# 06 — Generation & Prompt Engineering

Compare six prompt templates on the same retrieved context.

| Prompt | Style | Goal |
|---|---|---|
| `rag` | Grounded | Answer from context only |
| `cot` | Chain-of-Thought | Step-by-step reasoning |
| `react` | Reason + Act | Thought → Action → Answer |
| `tot` | Tree-of-Thought | Explore multiple approaches |
| `few_shot` | Examples | Show format via examples |
| `zero_shot` | Instructions only | Baseline |

**Config source:** `configs/default.yaml` -> `llm`, `notebooks.sample_question`
**Secrets source:** `.env` -> `HUGGINGFACEHUB_API_TOKEN`

In [ ]:
import sys
import os
sys.path.append("..")

from rag_pipeline.utils import load_notebook_config, format_docs
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.vectorstores import load_vectorstore
from rag_pipeline.retrieval import build_retriever
from rag_pipeline.generation import build_llms, PROMPT_REGISTRY

cfg, REPO = load_notebook_config()
FAISS_DIR = REPO / cfg.paths["faiss_index"]
QUESTION  = cfg.notebooks["sample_question"]

print("FAISS dir:", FAISS_DIR)
print("Question: ", QUESTION)

**Build retriever**

In [ ]:
emb = build_embeddings(dict(cfg.embeddings))
store = load_vectorstore(emb, {
    "type": "faiss",
    "persist_dir": str(FAISS_DIR),
})
retriever = build_retriever(store, {"search_type": "similarity", "k": 3})

**Load LLM**

In [ ]:
llms = build_llms(dict(cfg.llm))
model_name = next(iter(llms))
llm = llms[model_name]
print("Loaded model:", model_name)

In [ ]:
def render(prompt_name: str, question: str, max_chars: int = 300) -> str:
    template = PROMPT_REGISTRY[prompt_name]
    ctx = format_docs(retriever.invoke(question), max_chars=max_chars)
    prompt = template.format(context=ctx, question=question)
    return "".join(str(c) for c in llm.stream(prompt, max_new_tokens=200))

**Run all prompts**

In [ ]:
NAMES = ["rag", "cot", "react", "tot", "few_shot"]

for name in NAMES:
    print(f"\n{'='*60}\n### {name.upper()}\n{'='*60}")
    print(render(name, QUESTION))